# 08 — Embeddings y similitud semántica

**Level 0 — Fundamentos Software & IA**

Convertimos frases en vectores con `sentence-transformers`
(`all-MiniLM-L6-v2`) y medimos similitud con coseno: la base de la
búsqueda semántica.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

modelo = SentenceTransformer("all-MiniLM-L6-v2")
dimension = modelo.get_sentence_embedding_dimension()
print(f"   Modelo: {modelo}")
print(f"   Dimension del embedding: {dimension}")

/workspaces/student-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4216.12it/s]

   Modelo: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
   Dimension del embedding: 384


/tmp/ipykernel_84710/2085653841.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = modelo.get_sentence_embedding_dimension()


## Generar embeddings

In [2]:
frases = [
    "Los gatos son mascotas fantasticas",
    "Los perros son animales leales",
    "Me gusta programar en Python",
    "El lenguaje Python es muy versatil",
    "El clima hoy esta soleado",
]
embeddings = modelo.encode(frases)
print(f"   Frases: {len(frases)}")
print(f"   Embeddings shape: {embeddings.shape}")
print(f"\n   Embedding de '{frases[0]}':")
print(f"   Primeros 8 valores: {embeddings[0][:8]}")
print(f"   ... total {len(embeddings[0])} valores")

   Frases: 5
   Embeddings shape: (5, 384)

   Embedding de 'Los gatos son mascotas fantasticas':
   Primeros 8 valores: [ 0.0203064   0.03703362 -0.03381393 -0.06928898  0.00810382  0.00450937
  0.12006035  0.01854939]
   ... total 384 valores


## Matriz de similitud coseno

In [3]:
matriz_similitud = cosine_similarity(embeddings)
nombres = ["Gatos", "Perros", "Python1", "Python2", "Clima"]

print(f"   {'':>8}", end="")
for n in nombres:
    print(f"{n:>8}", end="")
print()
for i, n in enumerate(nombres):
    print(f"   {n:>8}", end="")
    for j in range(len(nombres)):
        print(f"{matriz_similitud[i][j]:>8.3f}", end="")
    print()

print("\n4. Interpretacion:")
for i in range(len(frases)):
    for j in range(i + 1, len(frases)):
        sim = matriz_similitud[i][j]
        nivel = (
            "MUY similar" if sim > 0.7
            else "similar" if sim > 0.5
            else "poco similar" if sim > 0.3
            else "diferente"
        )
        print(f"   {nombres[i]:>8} vs {nombres[j]:<8}: {sim:.3f} ({nivel})")

              Gatos  Perros Python1 Python2   Clima
      Gatos   1.000   0.579   0.266   0.236   0.409
     Perros   0.579   1.000   0.218   0.261   0.510
    Python1   0.266   0.218   1.000   0.630   0.382
    Python2   0.236   0.261   0.630   1.000   0.378
      Clima   0.409   0.510   0.382   0.378   1.000

4. Interpretacion:
      Gatos vs Perros  : 0.579 (similar)
      Gatos vs Python1 : 0.266 (diferente)
      Gatos vs Python2 : 0.236 (diferente)
      Gatos vs Clima   : 0.409 (poco similar)
     Perros vs Python1 : 0.218 (diferente)
     Perros vs Python2 : 0.261 (diferente)
     Perros vs Clima   : 0.510 (similar)
    Python1 vs Python2 : 0.630 (similar)
    Python1 vs Clima   : 0.382 (poco similar)
    Python2 vs Clima   : 0.378 (poco similar)


## Búsqueda semántica

In [4]:
consulta = "Los animales domesticos"
embedding_consulta = modelo.encode([consulta])
similitudes = cosine_similarity(embedding_consulta, embeddings)[0]
indice_mas_similar = np.argmax(similitudes)

print(f"   Consulta: '{consulta}'")
print(f"   Resultado: '{frases[indice_mas_similar]}'")
print(f"   Similitud: {similitudes[indice_mas_similar]:.3f}")
print("\n   Todas las similitudes:")
for i, frase in enumerate(frases):
    print(f"   {similitudes[i]:.3f} - {frase}")

   Consulta: 'Los animales domesticos'
   Resultado: 'Los perros son animales leales'
   Similitud: 0.756

   Todas las similitudes:
   0.545 - Los gatos son mascotas fantasticas
   0.756 - Los perros son animales leales
   0.257 - Me gusta programar en Python
   0.283 - El lenguaje Python es muy versatil
   0.523 - El clima hoy esta soleado


## Significado, no solo palabras

In [5]:
frases_similar = [
    "Me encanta la inteligencia artificial",
    "La IA me fascina profundamente",
    "Hoy desayune un cafe con leche",
]
emb_similar = modelo.encode(frases_similar)
sim_ia_cafe = cosine_similarity(emb_similar)

print(f"   'Me encanta la IA' vs 'La IA me fascina' = {sim_ia_cafe[0][1]:.3f}")
print(f"   'Me encanta la IA' vs 'cafe con leche'   = {sim_ia_cafe[0][2]:.3f}")

   'Me encanta la IA' vs 'La IA me fascina' = 0.489
   'Me encanta la IA' vs 'cafe con leche'   = 0.206


## Conclusión

- Los embeddings capturan **significado**, no coincidencia de palabras
- Frases semánticamente cercanas → coseno alto (~0.8-0.9)
- Temas distintos → coseno bajo
- La **búsqueda semántica**: la frase más cercana al vector de la consulta